In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


In [ ]:
import pandas as pd
import numpy as np

base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'
save_dir_plots = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Plots'


# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site!='UI']

# Let's get rid of the patients who do worse than chance on the DM task, or have no hits in the memory task

subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]

days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values


# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')

learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

# 3/18/2025

# recover the latent Q-values
learn_df['Q'] = learn_df.rpe + learn_df.reward
# compute the prior trial's rpe:
learn_df['prior_rpe'] = learn_df.groupby('participant')['rpe'].shift(1)
# compute the trials since last reversal
reversal_trials = [16, 33, 48, 65]
learn_df['trials_since_reversal'] = learn_df.apply(lambda x: np.mod(x.trials, reversal_trials[int(x.block-2)]) if x.block>1 else x.trials, axis=1)
# recode reward  as 0.5 and non-reward as -0.5
learn_df['preward'] = learn_df.apply(lambda x: 0.5 if x.reward==1 else -0.5 if x.reward==0 else np.nan, axis=1)
# recode choice==1 as 0.5 and choice==0 as -0.5
learn_df['pgamble'] = learn_df.apply(lambda x: 0.5 if x.gamble==1 else -0.5 if x.gamble==0 else np.nan, axis=1)

# Gelman standardization 
for subj in learn_df.participant.unique():

        learn_df.loc[learn_df.participant==subj, 
                'zQ'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'Q'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'Q'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'Q']))
        learn_df.loc[learn_df.participant==subj,
                'zprior_rpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'prior_rpe'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'prior_rpe'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'prior_rpe']))
        learn_df.loc[learn_df.participant==subj, 
                        'zrpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                                'rpe'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                        'rpe'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                        'rpe']))
        learn_df.loc[learn_df.participant==subj, 
                'zpm'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'DPRIME']))

        learn_df.loc[learn_df.participant==subj,
                'ztrials_since_reversal'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'trials_since_reversal'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'trials_since_reversal'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'trials_since_reversal']))
        # compute the reaction-time based EV with res[ect to each subject's min and max RT
        RT_inferred_EV_subj =  1 - ((learn_df.loc[learn_df.participant==subj, 
                                                     'gamble_rt_log'] - np.nanmin(learn_df.loc[learn_df.participant==subj, 
                                                                                                  'gamble_rt_log'])) / (np.nanmax(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                                  'gamble_rt_log']) - np.nanmin(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                                                                                'gamble_rt_log'])))
        # compute 
        learn_df.loc[learn_df.participant==subj,
                'RT_rpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'reward'] - RT_inferred_EV_subj)

        learn_df.loc[learn_df.participant==subj,
                'zRT_rpe'] = (learn_df.loc[learn_df.participant==subj, 
                                                                        'RT_rpe'] - np.nanmean(learn_df.loc[learn_df.participant==subj, 
                                                                                                'RT_rpe'])) / (2*np.nanstd(learn_df.loc[learn_df.participant==subj, 
                                                                                                                                'RT_rpe']))

        learn_df.loc[learn_df.participant==subj,
                'rpe_cat'] = pd.qcut(learn_df.loc[learn_df.participant==subj, 
                                                                                'rpe'], 3, labels=['neg', 'neutral', 'pos'])


In [ ]:
learn_df.rename(columns={'gamble': 'choice', 
'gamble_rt':'rt'}, inplace=True)

In [ ]:
# learn_df = learn_df[['participant', 'trials', 'choice', 'reward', 'rt']]

# Drop bad trials
learn_df = learn_df.dropna(subset=["choice", "reward", "rt"])
learn_df = learn_df[learn_df.rt > 0.25]      # remove anticipatory RTs
learn_df = learn_df[learn_df.rt < 10.0]       # remove outliers

# Convert participants to indices
learn_df["subj_idx"] = pd.factorize(learn_df["participant"])[0] + 1

In [ ]:
# Per-subject minimum RT (same order as subject indices 1..S) for t0 constraint
min_rt_subj = learn_df.groupby("subj_idx")["rt"].min().sort_index().values.astype(float)

stan_data = {
    "N": len(learn_df),
    "S": learn_df.subj_idx.nunique(),
    "subj": learn_df.subj_idx.values.astype(int),
    "choice": learn_df.choice.values.astype(int),
    "reward": learn_df.reward.values.astype(float),
    "rt": learn_df.rt.values.astype(float),
    "min_rt_subj": min_rt_subj,
    "max_a": 10.0,
}

In [ ]:
from cmdstanpy import CmdStanModel

model = CmdStanModel(stan_file="rl_ddm.stan")

In [ ]:
fit = model.sample(
    data=stan_data,
    chains=4,
    parallel_chains=4,
    iter_warmup=1000,
    iter_sampling=1000,
    adapt_delta=0.95,
    show_console=True
)


In [ ]:
import arviz as az
# After fit = model.sample(...)
idata = az.from_cmdstanpy(
    fit,
    log_likelihood="log_lik",   # only after adding log_lik in Stan (see below)
)
idata.to_netcdf(f"{load_dir}/rl_ddm_idata.nc")

In [ ]:
import arviz as az
from pathlib import Path

idata = az.from_cmdstanpy(fit, log_likelihood="log_lik")
Path("fits").mkdir(exist_ok=True)
idata.to_netcdf("fits/rl_ddm_idata.nc")
az.summary(idata, var_names=["mu_alpha", "mu_beta", "mu_a", "mu_t0"])

# Generated quantities are in idata.posterior (same as params) or sometimes in idata.posterior_predictive / idata.generated_quantity depending on version
drift_trial = idata.posterior["drift_trial"]   # xarray, dims typically (chain, draw, trial)
drift_mean_per_trial = drift_trial.mean(dim=["chain", "draw"]).values   # shape (N,)

# get all other trialwise parameters other than drift 



In [ ]:
az.summary(idata)

In [ ]:
import numpy as np

alpha_hat = idata.posterior["alpha"].mean(("chain", "draw")).values
beta_hat  = idata.posterior["beta"].mean(("chain", "draw")).values

Q = {s: np.array([0.5, 0.5]) for s in range(1, stan_data["S"] + 1)}
rpes = []

for i, row in learn_df.iterrows():
    s = row.subj_idx
    a = int(row.choice)
    r = row.reward

    rpe = r - Q[s][a]
    rpes.append(rpe)

    Q[s][a] += alpha_hat[s-1] * rpe

learn_df["rpe_ddm"] = rpes

# drift_trial has shape (n_draws, N); N matches len(learn_df) in same order
learn_df["drift_mean"] = fit.stan_variable("drift_trial").mean(axis=0)
# Optional: uncertainty
drift_std = fit.stan_variable("drift_trial").std(axis=0)
learn_df["drift_hdi_low"] = np.percentile(fit.stan_variable("drift_trial"), 2.5, axis=0)
learn_df["drift_hdi_high"] = np.percentile(fit.stan_variable("drift_trial"), 97.5, axis=0)


In [ ]:
import seaborn as sns
sns.regplot(data=learn_df, x="drift_mean", y="rpe_ddm", scatter_kws={"alpha": 0.1})

In [ ]:
from scipy.stats import pearsonr
pearsonr(learn_df["rpe"], learn_df["rpe_ddm"])


In [ ]:
#save out this learn_df
learn_df.to_csv(f'{load_dir}/learn_df_RWH_ddm.csv', index=False)

In [ ]:
import arviz as az

# If you saved NetCDF:
idata_rl_ddm = az.from_netcdf(f'{load_dir}/rl_ddm_idata.nc')
idata_drift_rpe = az.from_netcdf(f'{load_dir}/rl_ddm_driftRPE_stable_idata.nc')

# Compare (uses LOO by default)
comparison = az.compare(
    {"rl_ddm": idata_rl_ddm, "rl_ddm_driftRPE": idata_drift_rpe}
)
print(comparison)

In [ ]:
az.plot_compare(comparison)

2/13/2026: So from what I can tell, the vanilla RLDDM is better than the one with the drift-based RPE by a bit. 

So I should probably just leave the drift/RT based RPE stuff alone honestly. 